In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install datasets -q

In [ ]:
!pip install bert-score -q

In [ ]:
from unsloth import FastLanguageModel

Test Fonksiyonun Tanımlanması

In [ ]:
import torch
import re
import math
from tqdm import tqdm
from bert_score import score as bert_score
from google.colab import drive

test_dataset = load_dataset("json", data_files="test_sbertsiz.jsonl", split="train")

# 🔹 Puan çekme fonksiyonu
def extract_score(text):
    match = re.search(r"Puan:\s*(0\.\d+|1\.0+)", text)
    if match:
        return float(match.group(1))
    return None

# 🔹 Analiz metnini çekme fonksiyonu
def extract_analysis(text):
    # "Puan: 0.85" satırından önceki kısmı analiz olarak al
    parts = re.split(r"Puan:\s*[\d.]+", text)
    if parts:
        return parts[0].strip()
    return text.strip()


def evaluate_model_with_bertscore(model, tokenizer, dataset, max_length=2048, device="cuda"):

    total_loss = 0.0
    total_tokens = 0
    score_errors = []
    valid_score_count = 0

    # BERTScore için listeler
    all_predictions = []
    all_references  = []


    for example in tqdm(dataset, desc="Evaluating"):

        instruction = example["instruction"]
        input_text  = example["input"]
        target      = example["output"]

        prompt    = f"{instruction}\n{input_text}\n"
        full_text = prompt + target

        # 🔹 Tokenize + Label Masking
        inputs = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to(device)

        labels = inputs["input_ids"].clone()
        prompt_len = len(tokenizer(prompt)["input_ids"])
        labels[:, :prompt_len] = -100
        inputs["labels"] = labels

        # 🔹 Loss hesaplama
        with torch.no_grad():
            outputs = model(**inputs)

        num_tokens = (labels != -100).sum().item()
        total_loss   += outputs.loss.item() * num_tokens
        total_tokens += num_tokens

        # 🔹 Generate
        with torch.no_grad():
            generated_ids = model.generate(
                tokenizer(prompt, return_tensors="pt").input_ids.to(device),
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        # 🔹 MAE için puan karşılaştırma
        pred_score = extract_score(generated_text)
        true_score = extract_score(target)
        if pred_score is not None and true_score is not None:
            score_errors.append(abs(pred_score - true_score))
            valid_score_count += 1

        # 🔹 BERTScore için analiz metinlerini topla
        pred_analysis = extract_analysis(generated_text)
        true_analysis = extract_analysis(target)
        all_predictions.append(pred_analysis)
        all_references.append(true_analysis)

    # 🔹 Ortalama Loss & Perplexity
    avg_loss   = total_loss / total_tokens if total_tokens > 0 else 0
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")

    # 🔹 MAE
    avg_score_error = (
        sum(score_errors) / len(score_errors) if score_errors else None
    )

        # 🔹 BERTScore (tüm örnekler bittikten sonra toplu hesapla — daha hızlı)
    P, R, F1 = bert_score(
        all_predictions,
        all_references,
        lang="tr",
        device=device
    )
    avg_bertscore_p  = P.mean().item()
    avg_bertscore_r  = R.mean().item()
    avg_bertscore_f1 = F1.mean().item()

    return {
        "avg_loss"            : avg_loss,
        "perplexity"          : perplexity,
        "score_mae"           : avg_score_error,
        "bertscore_precision" : avg_bertscore_p,
        "bertscore_recall"    : avg_bertscore_r,
        "bertscore_f1"        : avg_bertscore_f1,
        "valid_score_samples" : valid_score_count,
        "total_samples"       : len(dataset)
    }

Untrained Qwen3 Modelinin Yüklenmesi

In [ ]:
"""QWEN 3 Normal"""

# ── Qwen3 Model Yükleme (Unsloth) ────────────────────────────────────────────
QWEN_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Yükleniyor: {QWEN_MODEL_NAME}")

qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = QWEN_MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,          # 4-bit quantization (BnbConfig'e gerek yok)
    dtype          = None,          # Otomatik tespit (bf16 destekliyorsa bf16)
)

FastLanguageModel.for_inference(qwen_model)
qwen_model.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    qwen_model, qwen_tokenizer, test_dataset
)

print("Qwen/Qwen3-4B-Instruct-2507 ---Untrained")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

Modelin Temizlenmesi

In [ ]:
import gc
del qwen_model, qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU belleği temizlendi ✓")
print(f"Serbest VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Fine-Tuned Modelin Yüklenmesi

In [ ]:
drive.mount('/content/drive')

lora_path_qwen3 = "/content/QWEN3_finetuned_eos_sbertsiz"

model_qwen3, tokenizer_qwen3 = FastLanguageModel.from_pretrained(
    model_name = lora_path_qwen3, # Kaydettiğimiz LoRA klasörü
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_qwen3)
model_qwen3.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    model_qwen3, tokenizer_qwen3, test_dataset
)
print("Qwen/Qwen3-4B-Instruct-2507 ---Fine-Tuned")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

Modelin Temizlenmesi

In [ ]:
import gc
del model_qwen3, tokenizer_qwen3
gc.collect()
torch.cuda.empty_cache()
print(f"GPU belleği temizlendi ✓")
print(f"Serbest VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Untrained QWEN 2.5 Modelin Yüklenmesi

In [ ]:
QWEN25_MODEL_NAME="Qwen/Qwen2.5-3B-Instruct"

qwen25_model_untrained, qwen25_tokenizer_untrained, = FastLanguageModel.from_pretrained(
    model_name     = QWEN25_MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,          # 4-bit quantization (BnbConfig'e gerek yok)
    dtype          = None,          # Otomatik tespit (bf16 destekliyorsa bf16)
)

FastLanguageModel.for_inference(qwen25_model_untrained)
qwen25_model_untrained.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    qwen25_model_untrained, qwen25_tokenizer_untrained, test_dataset
)
print("Qwen/Qwen2.5-3B-Instruct ---Untrained")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

Modelin Temizlenmesi

In [ ]:
import gc
del qwen25_model_untrained, qwen25_tokenizer_untrained
gc.collect()
torch.cuda.empty_cache()
print(f"GPU belleği temizlendi ✓")
print(f"Serbest VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Fine-Tuned QWEN 2.5 Modelin Yüklenmesi

In [ ]:
#lora_path_qwen25 = "/content/QWEN2.5_finetuned"
lora_path_qwen25 = "/content/QWEN2.5_finetuned"

qwen25_model, qwen25_tokenizer = FastLanguageModel.from_pretrained(
    model_name = lora_path_qwen25, # Kaydettiğimiz LoRA klasörü
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(qwen25_model)
qwen25_model.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    qwen25_model, qwen25_tokenizer, test_dataset
)

print("Qwen/Qwen2.5-3B-Instruct ---Fine-Tuned")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

Modelin Temizlenmesi

In [ ]:
import gc
del qwen25_model, qwen25_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU belleği temizlendi ✓")
print(f"Serbest VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")


Untrained OPENLLAMA Modelin Yüklenmesi

In [ ]:
OPENLLAMA_MODEL_NAME = "openlm-research/open_llama_3b"

openllama_model_untrained, openllama_tokenizer_untrained = FastLanguageModel.from_pretrained(
    model_name     = OPENLLAMA_MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,   # 4-bit quantization
    dtype          = None     # BF16 destekliyorsa otomatik seçilir
)

FastLanguageModel.for_inference(openllama_model_untrained)
openllama_model_untrained.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    openllama_model_untrained, openllama_tokenizer_untrained, test_dataset
)
print("openlm-research/open_llama_3b ---Untrained")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

Modelin Temizlenmesi

In [ ]:
del openllama_model_untrained, openllama_tokenizer_untrained
gc.collect()
torch.cuda.empty_cache()
print(f"GPU belleği temizlendi ✓")
print(f"Serbest VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Fine-Tuned OPENLLAMA Modelin Yüklenmesi

In [ ]:
lora_path_openllma = "/content/openllama_3b__finetuned_eos_sbertsiz"

openllma_model, openllma_tokenizer = FastLanguageModel.from_pretrained(
    model_name = lora_path_openllma, # Kaydettiğimiz LoRA klasörü
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(openllma_model)
openllma_model.eval()

Test

In [ ]:
results = evaluate_model_with_bertscore(
    openllma_model, openllma_tokenizer, test_dataset
)
print("openlm-research/open_llama_3b ---Fine-Tuned")
print(f"Loss        : {results['avg_loss']:.4f}")
print(f"Perplexity  : {results['perplexity']:.2f}")
print(f"MAE         : {results['score_mae']:.4f}")
print(f"BERTScore F1: {results['bertscore_f1']:.4f}")
print(f"BERTScore Precision: {results['bertscore_precision']:.4f}")
print(f"BERTScore Recall: {results['bertscore_recall']:.4f}")

In [ ]:
print(f"valid_score_samples: {results['valid_score_samples']:.4f}")